In [ ]:
# ==============================================================================
# PARALLEL DIM: MANAGER
# ==============================================================================
from helpers import IncrementalPipeline, TableConfig, get_latest_batch_id, setup_logger, safe_count, generate_batch_id
import pandas as pd
from helpers.silver_transforms import transform_manager_full_pipeline

logger = setup_logger("parallel_dim_manager")

batch_id = generate_batch_id()
pipeline = IncrementalPipeline(spark, dbutils, batch_id=batch_id)

dependencies = ["address", "city", "country"]

bronze_batch_id = get_latest_batch_id(spark, "staff")
if not bronze_batch_id:
    raise ValueError("No bronze batch_id found for staff; run bronze load first.")
logger.info(f"Using bronze batch_id for staff: {bronze_batch_id}")

config = TableConfig(
    table_name="staff",
    business_key="staff_id",
    surrogate_key="staff_key",
    watermark_column="last_update",
    scd_type=1,
    gold_table_name="dim_manager",
    silver_transform=transform_manager_full_pipeline,
    dependencies=dependencies,
)

print("Row Counts (Before):")
print(f"dim_manager: {safe_count(spark, 'dim_manager')}")
print("\nLatest Watermarks (Before):")
display(spark.table("wheelie.monitoring.watermarks"))

results = pipeline.load_tables([config], force_full=False, bronze_batch_id=bronze_batch_id)
display(pd.DataFrame(results))

print("\nRow Counts (After):")
print(f"dim_manager: {safe_count(spark, 'dim_manager')}")
print("\nLatest Watermarks (After):")
display(spark.table("wheelie.monitoring.watermarks"))
